# Material extra: filtrar e comparar proporções

Este notebook está **completo**, para estudo. Ele repete o roteiro das aulas 2 e
3 numa terceira base, e se aprofunda em duas coisas que ficaram apertadas em
aula: as maneiras de filtrar um DataFrame, e a comparação de proporções entre
grupos.

Rode célula a célula e, em cada bloco, tente prever o resultado antes de
executar.


In [ ]:
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

URL = "https://raw.githubusercontent.com/jtrecenti/202662-cdad2/main/dados"


## A pergunta de pesquisa

> Nas apelações do TJSP sobre negativa de cobertura de plano de saúde, com que
> frequência o acórdão discute dano moral, e isso varia entre as câmaras de
> Direito Privado?

A pergunta tem três recortes embutidos: só **apelações**, só **câmaras de
Direito Privado**, e o desfecho é **discutir dano moral**.


### De onde vieram os dados

```python
import juscraper as jus

tjsp = jus.scraper("tjsp")
acordaos = tjsp.cjsg('"plano de saude" E "negativa de cobertura"', paginas=range(1, 26))
```

Como na base da aula 3, as colunas `camara`, `secao`, `classe`, `assunto`,
`tem_dano_moral` e `n_palavras_ementa` foram lidas do texto antes da publicação.


In [ ]:
planos = pd.read_csv(f"{URL}/tjsp_cjsg_plano_saude.csv")
planos.head(3)


In [ ]:
planos.info()


## Preparando os tipos

Duas conversões da aula 2: a data e as categóricas.


In [ ]:
planos["data_julgamento"] = pd.to_datetime(planos["data_julgamento"], format="%d/%m/%Y")
planos["data_publicacao"] = pd.to_datetime(planos["data_publicacao"], format="%d/%m/%Y")

planos["dias_ate_publicacao"] = (
    planos["data_publicacao"] - planos["data_julgamento"]
).dt.days

planos["dias_ate_publicacao"].describe()


In [ ]:
planos["classe"] = pd.Categorical(planos["classe"])

planos["classe"].value_counts()


`camara` não veio preenchida em todas as linhas. Vale entender por quê antes de
filtrar: são os Núcleos de Justiça 4.0, que não são câmaras numeradas.


In [ ]:
planos["camara"].notna().mean().round(3)


In [ ]:
planos.loc[planos["camara"].isna(), "orgao_julgador"].value_counts().head()


A pergunta fala em câmaras, então esses casos vão sair do recorte **por
decisão**, e não por acidente.


## Filtrar: três jeitos de fazer a mesma coisa

### 1. Índice lógico

Uma comparação devolve uma série de `True` e `False` do mesmo tamanho do
DataFrame. É uma máscara, e não uma lista de posições:


In [ ]:
eh_apelacao = planos["classe"] == "Apelação Cível"

eh_apelacao.head()


In [ ]:
len(eh_apelacao), eh_apelacao.sum()


In [ ]:
planos[eh_apelacao].shape


Combinando condições: `&` é "e", `|` é "ou", `~` é "não". Os parênteses são
obrigatórios, porque `&` tem precedência maior que `==`. Sem eles o Python tenta
avaliar a coisa errada e levanta erro, como no terceiro exemplo abaixo.


In [ ]:
sp_com_dano = planos[(planos["comarca"] == "São Paulo") & planos["tem_dano_moral"]]

len(sp_com_dano)


In [ ]:
sem_camara = planos[~planos["camara"].notna()]

len(sem_camara)


In [ ]:
try:
    planos[planos["comarca"] == "São Paulo" & planos["tem_dano_moral"]]
except TypeError as erro:
    print("TypeError:", erro)


Três auxiliares que evitam condições longas:


In [ ]:
pd.Series({
    "isin": len(planos[planos["comarca"].isin(["São Paulo", "Campinas", "Santos"])]),
    "between": len(planos[planos["dias_ate_publicacao"].between(0, 7)]),
    "notna": len(planos[planos["camara"].notna()]),
})


### 2. `.loc`

`.loc[linhas, colunas]` filtra e escolhe colunas na mesma expressão:


In [ ]:
planos.loc[eh_apelacao, ["processo", "camara", "secao", "tem_dano_moral"]].head()


E é a forma que você **precisa** usar para atribuir. O `df[cond]["coluna"] = x`
mexe numa cópia temporária, então a alteração se perde:


In [ ]:
copia = planos.copy()
copia[copia["comarca"] == "Santos"]["comarca"] = "SANTOS"

(copia["comarca"] == "SANTOS").sum()


In [ ]:
copia.loc[copia["comarca"] == "Santos", "comarca"] = "SANTOS"

(copia["comarca"] == "SANTOS").sum()


`.loc` fatia por **rótulo** do índice, e `.iloc` fatia por **posição**. A
diferença aparece no limite superior, que o `.loc` inclui e o `.iloc` não:


In [ ]:
pd.Series({
    ".loc[0:3]": len(planos.loc[0:3]),
    ".iloc[0:3]": len(planos.iloc[0:3]),
})


### 3. `.query`

`.query` recebe a condição escrita como texto, e usa `@` para referir uma
variável do Python:


In [ ]:
planos.query("classe == 'Apelação Cível' and n_palavras_ementa > 150").shape


In [ ]:
limite = 150

planos.query("n_palavras_ementa > @limite and secao == 'Direito Privado'").shape


Os três jeitos dão o mesmo resultado:


In [ ]:
a = planos[(planos["classe"] == "Apelação Cível") & (planos["secao"] == "Direito Privado")]
b = planos.loc[(planos["classe"] == "Apelação Cível") & (planos["secao"] == "Direito Privado")]
c = planos.query("classe == 'Apelação Cível' and secao == 'Direito Privado'")

len(a), len(b), len(c)


Índice lógico é o mais explícito, `.loc` é o único seguro para atribuir, e
`.query` é o mais legível com muitas condições. Uma limitação do `.query`: nomes
de coluna com espaço ou acento precisam de crase, e nem toda expressão do pandas
funciona lá dentro.


### O recorte da pergunta


In [ ]:
apelacoes = planos.query(
    "classe == 'Apelação Cível' and secao == 'Direito Privado'"
).dropna(subset=["camara"]).copy()

len(planos), len(apelacoes)


## Proporções comparadas entre grupos

`tem_dano_moral` é binária, então a média é a proporção:


In [ ]:
apelacoes["tem_dano_moral"].mean().round(3)


In [ ]:
apelacoes["tem_dano_moral"].value_counts(normalize=True).round(3)


Para variável nominal, o que existe é contagem e proporção:


In [ ]:
apelacoes["camara"].value_counts().head(8)


`pd.crosstab` faz a tabela de duas entradas. Sem `normalize`, ela traz
contagens:


In [ ]:
pd.crosstab(apelacoes["camara"], apelacoes["tem_dano_moral"]).head()


Com `normalize="index"`, cada linha soma 1, e a tabela passa a mostrar a
proporção dentro de cada câmara:


In [ ]:
pd.crosstab(
    apelacoes["camara"], apelacoes["tem_dano_moral"], normalize="index"
).round(3).head()


`normalize="columns"` divide pela coluna e `normalize=True` divide pelo total
geral. Trocar um pelo outro muda completamente a leitura, e é um erro comum em
relatório. Compare:


In [ ]:
pd.crosstab(apelacoes["secao"], apelacoes["tem_dano_moral"], normalize="index").round(3)


In [ ]:
pd.crosstab(apelacoes["secao"], apelacoes["tem_dano_moral"], normalize=True).round(3)


Uma câmara com poucos acórdãos tem proporção instável, e mostrar isso ao lado
das outras engana. Vale sempre olhar a contagem junto da proporção:


In [ ]:
contagem = apelacoes["camara"].value_counts()
proporcao = pd.crosstab(
    apelacoes["camara"], apelacoes["tem_dano_moral"], normalize="index"
)[True]

pd.DataFrame({"n": contagem, "proporcao": proporcao.round(3)}).sort_values(
    "proporcao", ascending=False
).head(10)


## Antes de concluir qualquer coisa

A tabela acima mostra proporções que vão de cerca de 0,20 a 0,52. Antes de dizer
que as câmaras decidem diferente, considere:

1. **A medida é grosseira.** `tem_dano_moral` marca a ementa que *menciona* dano
   moral, inclusive para negar. Uma câmara que escreve ementa mais longa tende a
   mencionar mais coisas.
2. **A distribuição de casos não é aleatória.** Cada câmara recebe um perfil
   diferente de recurso, por sorteio, por prevenção e por comarca de origem.
3. **O tamanho do grupo importa.** Com 13 acórdãos, uma proporção de 0,46 muda
   para 0,38 se um único caso for classificado diferente.
4. **A amostra é o que a busca devolveu**, e não todos os acórdãos do tribunal
   sobre o tema.

Nenhum desses pontos invalida o exercício. Todos precisam estar escritos no
relatório.


## Para praticar

1. Refaça a tabela por câmara usando apenas acórdãos julgados em 2026.
2. Compare a mediana de `n_palavras_ementa` entre as câmaras com maior e menor
   proporção de menção a dano moral. Elas escrevem ementas de tamanhos
   diferentes?
3. A base tem uma coluna `valor_indenizacao`, preenchida em cerca de um quarto
   das linhas. Monte o recorte com valor e calcule mediana e IQR.


In [ ]:
# espaço para praticar
